In [1]:
import pandas as pd
import numpy as np
import json


In [2]:
df = pd.read_csv('receipt.csv', sep=',')

df


,uuid,started_at,status,response,skus,loaded
0,8421e51a-f919-4dd5-a5c2-8f0b558032ca,2024-03-03 22:59:17.333222,completed,"{""id"": 5178902281095588352, ""ofdId"": ""ofd5"", ""...","[{""SKU"": ""air_optix_plus_hydraglyde_multifocal...",manual
1,c5be9fc3-f5b6-4bee-b180-2dd1b7e287e5,2024-07-22 05:20:29.966083,completed,"{""id"": 5393036590092313857, ""ofdId"": ""ofd4"", ""...","[{""SKU"": ""air_optix_aqua_multifocal_3"", ""error...",manual
2,ccfc9ca5-dffe-4c45-a440-704fa49cab1a,2024-07-22 06:39:47.649944,completed,"{""id"": 5395792045137564928, ""ofdId"": ""ofd9"", ""...","[{""SKU"": ""dailies_total1_30"", ""errorCode"": nul...",scan
3,2c8c62d1-d18b-4925-ae93-af1df0648484,2024-07-22 06:44:24.156876,completed,"{""id"": 5388525829575596544, ""ofdId"": ""ofd9"", ""...","[{""SKU"": ""air_optix_plus_hydraglyde_3"", ""error...",manual
4,5a7c60ab-c660-4d62-831c-0bc2ef3c3b8a,2024-07-22 06:46:28.381774,completed,"{""id"": 5395796719653353728, ""ofdId"": ""ofd9"", ""...","[{""SKU"": ""dailies_total1_90"", ""errorCode"": nul...",scan
...,...,...,...,...,...,...
112,e6bf6209-38e3-4ecc-8fe3-549c5a888848,2024-07-26 21:43:18.004217,completed,"{""id"": 5402044229447317760, ""ofdId"": ""ofd4"", ""...",[],scan
113,8f534e69-8964-4a00-93be-987fb8092f04,2024-07-27 09:11:40.443845,completed,"{""id"": 5403280550508476416, ""ofdId"": ""ofd4"", ""...",[],scan
114,476e97b7-012c-4bc2-b6c1-b37b185357ce,2024-07-28 16:25:49.950148,completed,"{""id"": 5388190784373840128, ""ofdId"": ""ofd9"", ""...",[],scan
115,9ed846d6-119b-4d94-8014-b6c51242fd72,2024-07-28 18:56:21.316999,completed,"{""id"": 5405319890084411392, ""ofdId"": ""ofd5"", ""...",[],scan


Готовые отчеты для обработки находятся в папке "Отчет по нейросети"

In [3]:
#Сроки без ответа нейросети

smp_dict = {
    'started_at': [],
    'uuid': [],
    'response': [],
    'name': [],
    'source': []
    
}

#Берем только завершенные строки, где список skus пустой

for key, row in df[(df.status == 'completed') & (df.skus == '[]')].iterrows():
    
    resp = json.loads(row.response) #превращаю json-строку из response в словрик
    st_dat = row['started_at']
    
    # В одном response может быть несколько товаров, поэтому проходим по каждому
    
    for items in resp['content']['items']:
        
        smp_dict['started_at'].append(st_dat)
        smp_dict['uuid'].append(row.uuid)
        smp_dict['response'].append(resp)
        smp_dict['source'].append(row.loaded)
        
        # у некоторых товаров поле name может отсутствовать
        
        try:
            smp_dict['name'].append(items['name'])
        except KeyError:
            smp_dict['name'].append(None)
            
            
res = pd.DataFrame.from_dict(smp_dict)

# Добавляем пустые поля, чтобы структура отчета совпадала с основным отчетом

res['raw_json'] = None
res['sku (prediction)'] = None
res['errorCode'] = None

cols = [
    'started_at',
    'uuid',
    'response',
    'raw_json',
    'name',
    'sku (prediction)',
    'errorCode',
    'source'
]

res = res[cols]

res.to_excel('./Отчет по нейросети/result_null.xlsx', index = False)



#Сроки c ответом нейросети

mp_dict = {
    'started_at': [],
    'uuid': [],
    'response': [],
    'raw_json': [],
    'name': [],
    'sku (prediction)': [],
    'errorCode': [],
    'source': []
    
} 


# Оставляем завершенные запросы, где skus не пустой и не NaN

for key, row in df[(df.status == 'completed') & (df.skus != '[]') & (df.skus.notnull())].iterrows():
    
    resp = json.loads(row.response) #превращаю json-строку из response в словрик
    st_dat = row['started_at']
    
    try:
        
        pred = json.loads(row.skus) #разбираем предсказания нейросети из JSON
        
        #Одному запросу ...
        
        for items in pred:
            mp_dict['started_at'].append(st_dat)
            mp_dict['uuid'].append(row.uuid)
            mp_dict['response'].append(resp)
            mp_dict['raw_json'].append(pred)
            mp_dict['name'].append(items['name'])
            mp_dict['sku (prediction)'].append(items['SKU'])
            mp_dict['source'].append(row.loaded)
            
            
            #errorCode приходят не во всех ответах
            try:
                mp_dict['errorCode'].append(items['errorCode'])
            except KeyError:
                mp_dict['errorCode'].append(None)
                
                
    #Если значение skus невозможно разобрать как ожидаемый JSON
    
    except TypeError:
        
        mp_dict['started_at'].append(st_dat)
        mp_dict['uuid'].append(row.uuid)
        mp_dict['response'].append(resp)
        mp_dict['raw_json'].append('NaN')
        mp_dict['name'].append('NaN')
        mp_dict['sku (prediction)'].append('NaN')
        mp_dict['source'].append('NaN')
        

res2 = pd.DataFrame.from_dict(mp_dict)
res2.to_excel('./Отчет по нейросети/result_not_null.xlsx', index = False)
        
        
        
        